In [22]:
! pip install folium

## Importation et mise en forme

In [23]:
from astrovision.plot.plot_utils import make_mosaic
from astrovision.data import SatelliteImage, SegmentationLabeledSatelliteImage
import s3fs
import os
import geopandas as gpd
from pyproj import CRS
import matplotlib.pyplot as plt
import pandas as pd
import folium
import numpy as np
import math
import ipywidgets as widgets
from ipywidgets import interact

from IPython.display import clear_output

In [24]:
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://" + "minio.lab.sspcloud.fr"})
out  = fs.download(rpath="projet-slums-detection/ilots", lpath="ilots", recursive=True)
out = fs.download(rpath="projet-slums-detection/data-prediction/PLEIADES/MAYOTTE", lpath="pred_mayotte", recursive=True)

pred_2020 = gpd.read_file('pred_mayotte/2020/test/15/predictions.gpkg')
pred_2023 = gpd.read_file('pred_mayotte/2023/test/15/predictions.gpkg')
ilots = gpd.read_file("ilots/ilots.gpkg")
tgt_crs = CRS.from_epsg(4471) # mayotte 

pred_2020 = pred_2020.to_crs(tgt_crs)
pred_2023 = pred_2023.to_crs(tgt_crs)

pred_2020["area"] = pred_2020.area
pred_2023["area"] = pred_2023.area

ilots = ilots.to_crs(tgt_crs)
ilotsMayotte = ilots[ilots["ident_ilot"].str.startswith('976')]
ilotsMayotte["area"] = ilotsMayotte.geometry.area

pred_2020 = pred_2020.rename(columns={'area': 'aire_bati'})
pred_2023 = pred_2023.rename(columns={'area': 'aire_bati'})

/opt/mamba/lib/python3.11/site-packages/geopandas/geodataframe.py:1525: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [25]:
intersection_2020 = gpd.overlay(pred_2020, ilotsMayotte, how='intersection')
intersection_2023 = gpd.overlay(pred_2023, ilotsMayotte, how='intersection')

In [26]:
intersection_2020['aire_bati_par_ilot'] = intersection_2020.groupby('ident_ilot')['aire_bati'].transform('sum')
intersection_2020

,filename,id,aire_bati,ident_ilot,code,depcom_2018,ident_up,area,geometry,aire_bati_par_ilot
0,projet-slums-detection/data-raw/PLEIADES/MAYOT...,0,44.25,976120305,0305,97612,UP000000,4.329483e+06,"POLYGON ((502604.500 8601994.500, 502604.500 8...",28226.75
1,projet-slums-detection/data-raw/PLEIADES/MAYOT...,1,41.00,976120305,0305,97612,UP000000,4.329483e+06,"POLYGON ((502550.000 8601986.500, 502550.000 8...",28226.75
2,projet-slums-detection/data-raw/PLEIADES/MAYOT...,2,27.25,976120305,0305,97612,UP000000,4.329483e+06,"POLYGON ((502601.000 8601974.500, 502601.000 8...",28226.75
3,projet-slums-detection/data-raw/PLEIADES/MAYOT...,3,0.75,976120305,0305,97612,UP000000,4.329483e+06,"POLYGON ((502592.500 8601970.500, 502592.500 8...",28226.75
4,projet-slums-detection/data-raw/PLEIADES/MAYOT...,4,79.50,976120305,0305,97612,UP000000,4.329483e+06,"POLYGON ((502606.000 8601981.000, 502606.000 8...",28226.75
...,...,...,...,...,...,...,...,...,...,...
28264,projet-slums-detection/data-raw/PLEIADES/MAYOT...,27,761.00,976040601,0601,97604,UP000000,3.562064e+06,"POLYGON ((511045.000 8569037.000, 511045.000 8...",14086.25
28265,projet-slums-detection/data-raw/PLEIADES/MAYOT...,0,22.50,976160204,0204,97616,UP000000,8.522999e+05,"POLYGON ((513592.500 8578918.000, 513592.500 8...",15145.00
28266,projet-slums-detection/data-raw/PLEIADES/MAYOT...,1,7.00,976160204,0204,97616,UP000000,8.522999e+05,"POLYGON ((513500.500 8578631.000, 513500.500 8...",15145.00
28267,projet-slums-detection/data-raw/PLEIADES/MAYOT...,2,173.25,976160205,0205,97616,UP000000,9.368580e+05,"POLYGON ((513084.500 8578482.500, 513084.500 8...",9829.00


## Traitement des suppressions et ajouts

In [27]:
sym_diff = gpd.overlay(pred_2020, pred_2023, how='symmetric_difference')
index = pd.RangeIndex(stop=len(sym_diff))
sym_diff = gpd.GeoDataFrame(sym_diff, crs='EPSG:4471', index=index)

polygones_commun = gpd.overlay(pred_2020, pred_2023, how='intersection')
resultat = sym_diff[~sym_diff.geometry.isin(polygones_commun.geometry)]

In [ ]:
resultat_2020 = gpd.sjoin(resultat, pred_2020, how='left', op='within')
suppression = resultat_2020[resultat_2020.index_right.isna()]
suppression = suppression.loc[:, resultat.columns]
#suppression = suppression.drop('area_1', axis=1)

resultat_2023 = gpd.sjoin(resultat, pred_2023, how='left', op='within')
creation = resultat_2023[resultat_2023.index_right.isna()]
creation = creation.loc[:, resultat.columns]
#creation = creation.drop('area_1', axis=1)

/opt/mamba/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3517: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


/opt/mamba/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3517: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


In [ ]:
# intersection ilots creation destruction
ilots_inter_creation = gpd.overlay(creation, ilotsMayotte, how='intersection')
ilots_inter_creation["type"] = "creation"

ilots_inter_suppression = gpd.overlay(suppression, ilotsMayotte, how='intersection')
ilots_inter_suppression["type"] = "suppression"

df_diff = pd.concat([ilots_inter_creation, ilots_inter_suppression])
df_diff['compacite'] = (4 * math.pi * df_diff.area) / (df_diff.length**2)
df_diff = df_diff.drop(columns=['filename_1', 'filename_2','code','depcom_2018','ident_up','aire_bati_1','aire_bati_2'])
df_diff["aire_bati"] = df_diff["geometry"].area

## Nettoyage des tables

In [ ]:
def filtre_compacite (table, seuil_compacite):
    table['compacite'] = (4 * math.pi * table.area) / (table.length**2)
    table_filtree = table[table['compacite'] > seuil_compacite]
    return table_filtree

def filtre_taille (table, seuil_taille):
    if seuil_taille != 0:
        table_triee = table.sort_values(by='aire_bati')
        decile_seuil = np.percentile(table_triee['aire_bati'], seuil_taille)
        polygone_decile = table_triee[table_triee['air_ebati'] <= decile_seuil].iloc[-1]
        table_filtree = table[table['aire_bati'] > polygone_decile['aire_bati']]
    else : 
        table_filtree = table
    return table_filtree

In [ ]:
seuil_compacite = 0.08
seuil_taille = 10

In [ ]:
df_diff = filtre_taille(df_diff,seuil_taille)
df_diff = filtre_compacite(df_diff,seuil_compacite)

## Création de la table par îlot avec les aires

In [ ]:
df_diff["aireplus"] = df_diff.groupby('ident_ilot').filter(lambda x: (x['type'] == "creation").any())['aire_bati'].sum()
df_diff["airemoins"] = df_diff.groupby('ident_ilot').filter(lambda x: (x['type'] == "suppression").any())['aire_bati'].sum()

In [ ]:
somme_aireplus = df_diff.loc[df_diff['type'] == 'creation'].groupby('ident_ilot')['aire_bati'].sum()
df_diff['aireplus'] = df_diff['ident_ilot'].map(somme_aireplus)
somme_airemoins = df_diff.loc[df_diff['type'] == 'suppression'].groupby('ident_ilot')['aire_bati'].sum()
df_diff['airemoins'] = df_diff['ident_ilot'].map(somme_airemoins)
df_diff['airetotale'] = df_diff['aireplus'] - df_diff['airemoins']

In [ ]:
intersection_2020_unique = intersection_2020.drop_duplicates(subset=['ident_ilot'])
intersection_2020_unique = gpd.GeoDataFrame(intersection_2020_unique, geometry='geometry')
df_diff = df_diff.merge(intersection_2020_unique[['ident_ilot','aire_bati_par_ilot']], on='ident_ilot', how='left')

In [ ]:
df_diff['aire_plus_pct'] = 100 * df_diff['aireplus'] / df_diff['aire_bati_par_ilot']
df_diff['aire_moins_pct'] = 100 * df_diff['airemoins'] / df_diff['aire_bati_par_ilot']
df_diff['aire_totale_pct'] = 100 * df_diff['airetotale'] / df_diff['aire_bati_par_ilot']

## Export de la table

In [ ]:
df_diff_ilot = df_diff.drop_duplicates(subset=['ident_ilot'])
df_diff_ilot = df_diff_ilot.drop(columns=['area','geometry','type','compacite','id_1','id_2'])
df_diff_ilot = df_diff_ilot.merge(ilotsMayotte[['ident_ilot','area','geometry']], on='ident_ilot', how='right')
df_diff_ilot = gpd.GeoDataFrame(df_diff_ilot, geometry="geometry")

In [ ]:
df_diff_ilot.to_file('stats_ilots.gpkg', driver='GPKG')